<a href="https://colab.research.google.com/github/Angel1323x/Anwendungsprojekt-Verkehrsunfaelle/blob/main/ML_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import pandas as pd
import os


drive.mount('/content/drive')
target_path = '/content/drive/MyDrive/Anwendungsprojekt Colab - Unfall /Training_Test_Sets'

X_train = pd.read_parquet(os.path.join(target_path, 'X_train.parquet'))
X_test = pd.read_parquet(os.path.join(target_path, 'X_test.parquet'))
y_train = pd.read_parquet(os.path.join(target_path, 'y_train.parquet'))['CRASH_SEVERITY_LEVEL']
y_test = pd.read_parquet(os.path.join(target_path, 'y_test.parquet'))['CRASH_SEVERITY_LEVEL']

print(" Daten im neuen Notebook einsatzbereit!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Daten im neuen Notebook einsatzbereit!


# XGBoost


## Klassengewichtung (Class Weights)

Wir übergeben dem Modell ein Gewicht für jede Klasse, das umgekehrt proportional zu ihrer Häufigkeit ist.
Die Klassengewichte werden automatisch berechnet und ein erstes XGBoost-Baseline-Modell wird trainiert.[Linktext](https://)

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Identify and convert object columns to category dtype
# This ensures XGBoost can process them when enable_categorical=True
for col in ['TRAFFIC_CONTROL_DEVICE', 'LOCATION']:
    if col in X_train.columns and X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype('category')
    if col in X_test.columns and X_test[col].dtype == 'object':
        X_test[col] = X_test[col].astype('category')

# Also ensure 'SPEED_CATEGORY' is treated as category if it's not already
if 'SPEED_CATEGORY' in X_train.columns and X_train['SPEED_CATEGORY'].dtype != 'category':
    X_train['SPEED_CATEGORY'] = X_train['SPEED_CATEGORY'].astype('category')
if 'SPEED_CATEGORY' in X_test.columns and X_test['SPEED_CATEGORY'].dtype != 'category':
    X_test['SPEED_CATEGORY'] = X_test['SPEED_CATEGORY'].astype('category')

# 1. Klassengewichte berechnen (y_train muss ein 1D-Array/Spalte sein)
# Da es sich um Multiklassifikation handelt, mappen wir die Gewichte auf die einzelnen Zeilen
y_train_flat = y_train.values.ravel() if hasattr(y_train, 'values') else y_train

classes = np.unique(y_train_flat)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_flat)
class_weights_dict = dict(zip(classes, weights))

print("Berechnete Klassengewichte (Invers zur Häufigkeit):", class_weights_dict)

# Erstelle ein Array von Gewichten für jedes sample im Trainingsset
sample_weights = np.array([class_weights_dict[cls] for cls in y_train_flat])

# 2. XGBoost Classifier definieren
# 'multi:softprob' gibt uns die Wahrscheinlichkeiten für alle 4 Klassen
model = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    tree_method='hist',
    enable_categorical=True # Hinzugefügt, um kategoriale Features zu ermöglichen
)

# 3. Modell trainieren mit Sample Weights
print("Starte Modelltraining...")
model.fit(X_train, y_train_flat, sample_weight=sample_weights)
print("Training abgeschlossen.")

# 4. Vorhersagen generieren
y_test_flat = y_test.values.ravel() if hasattr(y_test, 'values') else y_test
y_pred = model.predict(X_test)

# 5. Evaluation
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test_flat, y_pred, target_names=['Blechschaden (0)', 'Leicht (1)', 'Schwer (2)', 'Tödlich (3)']))

# Konfusionsmatrix plotten
cm_normalized = confusion_matrix(y_test_flat, y_pred, normalize='true')
plt.figure(figsize=(8, 6))

sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['Pred 0', 'Pred 1', 'Pred 2', 'Pred 3'],
            yticklabels=['True 0', 'True 1', 'True 2', 'True 3'])
plt.ylabel('Tatsächliche Klasse (True)')
plt.xlabel('Vorhergesagte Klasse (Pred)')
plt.title('Normalized Confusion Matrix (Recall per Class)')
plt.show()

Berechnete Klassengewichte (Invers zur Häufigkeit): {np.int64(0): np.float64(0.297892689841929), np.int64(1): np.float64(1.728995253164557), np.int64(2): np.float64(16.49811273276296), np.int64(3): np.float64(243.73048327137548)}
Starte Modelltraining...


## Multiklassifikation in binäre Klassifikation
Es wird die 4-Klassen-Multiklassifikation in ein binäres Klassifikation umwandeln (0 = Reiner Blechschaden vs. 1 = Personenschaden).
Wenn wir die Klassen 1 (9.480), 2 (994) und 3 (67) zusammenfassen, kommen wir auf insgesamt 10.541 Fälle von Personenschaden.

In [ ]:
# 1. Daten kopieren und die Zielvariable binarisieren
y_train_bin = y_train.copy()
y_test_bin = y_test.copy()

if isinstance(y_train_bin, pd.DataFrame) or isinstance(y_train_bin, pd.Series):
    y_train_bin = (y_train_bin.iloc[:, 0] if isinstance(y_train_bin, pd.DataFrame) else y_train_bin).apply(lambda x: 1 if x > 0 else 0)
    y_test_bin = (y_test_bin.iloc[:, 0] if isinstance(y_test_bin, pd.DataFrame) else y_test_bin).apply(lambda x: 1 if x > 0 else 0)
else:
    y_train_bin = np.where(y_train_bin > 0, 1, 0)
    y_test_bin = np.where(y_test_bin > 0, 1, 0)

# 2. Entfernung der fehlerhaften Konvertierung zu float
# Die Spalten 'TRAFFIC_CONTROL_DEVICE', 'LOCATION' und 'SPEED_CATEGORY'
# wurden bereits in der vorherigen Zelle in 'category' konvertiert
# und XGBoost kann diese mit `enable_categorical=True` direkt verarbeiten.

# 3. Berechnen von scale_pos_weight
num_neg = np.sum(y_train_bin == 0)
num_pos = np.sum(y_train_bin == 1)
scale_pos_weight_value = num_neg / num_pos

print("Verteilung im Training (0 = Blech, 1 = Person):")
print(y_train_bin.value_counts() if hasattr(y_train_bin, 'value_counts') else np.bincount(y_train_bin))
print(f"Gewichtungsfaktor (scale_pos_weight): {scale_pos_weight_value:.2f}\n")

# 4. Binäres XGBoost Modell mit maximaler Fehlertoleranz definieren
binary_model = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight_value,
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    tree_method='hist',
    enable_categorical=True
)

print("Starte binäres Modelltraining...")
# Wir fitten das Modell mit den originalen X_train Daten, die bereits korrekt kategorisiert wurden.
binary_model.fit(X_train, y_train_bin)
print("Training abgeschlossen.")

# 5. Vorhersagen & Evaluation
y_pred_bin = binary_model.predict(X_test)

print("\n--- BINARY CLASSIFICATION REPORT ---")
print(classification_report(y_test_bin, y_pred_bin, target_names=['Blechschaden (0)', 'Personenschaden (1)']))

# 6. Normalisierte Konfusionsmatrix plotten
cm_bin = confusion_matrix(y_test_bin, y_pred_bin, normalize='true')
plt.figure(figsize=(7, 5))
sns.heatmap(cm_bin, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=['Pred Blech (0)', 'Pred Person (1)'],
            yticklabels=['True Blech (0)', 'True Person (1)'])
plt.ylabel('Tatsächliche Klasse')
plt.xlabel('Vorhergesagte Klasse')
plt.title('Normalized Confusion Matrix (Recall) - Binäres Modell')
plt.show()

## Hyperparameter-Tuning
Beim Hyperparameter-Tuning von XGBoost bei unbalancierten Daten wollen wir vor allem verhindern, dass das Modell die positive Klasse (Personenschaden) einfach nur "auswendig lernt" (Overfitting) oder durch zu aggressive Gewichtung zu viele Fehlalarme produziert. Die konkrete Methode nennt sich Grid Search.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Parameter-Gitter (Grid) definieren
# Wir halten es bewusst kompakt, damit Colab nicht stundenlang rechnet
param_grid = {
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 5, 10],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

# 2. Basis-Modell initialisieren
# scale_pos_weight_value haben wir ja bereits im vorherigen Schritt berechnet
base_binary_model = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight_value,
    n_estimators=100,
    tree_method='hist',
    enable_categorical=True,
    random_state=42
)

# 3. GridSearchCV aufsetzen
# Wir optimieren auf 'f1_macro', um die Minderheitenklasse stark zu berücksichtigen
# cv=3 reicht für eine universitäre Baseline völlig aus und spart Zeit
grid_search = GridSearchCV(
    estimator=base_binary_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=2,
    n_jobs=-1  # Nutzt alle verfügbaren CPU-Kerne in Colab
)

print("Starte Hyperparameter-Tuning (Grid Search)...")
grid_search.fit(X_train, y_train_bin)
print("Tuning abgeschlossen.")

# 4. Beste Parameter ausgeben
print("\nBeste Parameter-Kombination:")
print(grid_search.best_params_)
print(f"Bester Macro F1-Score im Training: {grid_search.best_score_:.4f}")

# 5. Das beste Modell für die Testdaten verwenden
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

# 6. Evaluation des getunten Modells
print("\n--- TUNED BINARY CLASSIFICATION REPORT ---")
print(classification_report(y_test_bin, y_pred_tuned, target_names=['Blechschaden (0)', 'Personenschaden (1)']))

# 7. Neue Konfusionsmatrix plotten
cm_tuned = confusion_matrix(y_test_bin, y_pred_tuned, normalize='true')
plt.figure(figsize=(7, 5))
sns.heatmap(cm_tuned, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=['Pred Blech (0)', 'Pred Person (1)'],
            yticklabels=['True Blech (0)', 'True Person (1)'])
plt.ylabel('Tatsächliche Klasse')
plt.xlabel('Vorhergesagte Klasse')
plt.title('Normalized Confusion Matrix (Recall) - Getuntes Modell')
plt.show()

## Feature Importance
XGBoost bringt eine eingebaute Metrik mit, die misst, wie oft und wie effektiv eine bestimmte Variable (z. B. eine Baustelle oder eine Geschwindigkeitszone) genutzt wurde, um die Daten zu splitten.
Sie sagt dir nur, dass ein Feature wichtig ist, aber nicht, in welche Richtung (z. B. ob Regen das Risiko erhöht oder senkt).

In [ ]:
# 1. Feature Importances aus dem besten Modell der Grid Search ziehen
importances = best_model.feature_importances_
feature_names = X_train.columns

# 2. In ein DataFrame packen und nach Wichtigkeit sortieren
feature_imp_df = pd.DataFrame({
    'Faktor (Feature)': feature_names,
    'Wichtigkeit (Gini Importance)': importances
}).sort_values(by='Wichtigkeit (Gini Importance)', ascending=False)

# 3. Grafik anzeigen (Die Top 15 Faktoren)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(11, 7))
sns.barplot(
    x='Wichtigkeit (Gini Importance)',
    y='Faktor (Feature)',
    data=feature_imp_df.head(15),
    palette='magma'
)
plt.title('Welche Faktoren treiben das Risiko für Personenschäden? (Top 15)', fontsize=14)
plt.xlabel('Relative Wichtigkeit im Modell', fontsize=12)
plt.ylabel('Unfall-Faktor', fontsize=12)
plt.tight_layout()
plt.show()

# 4. Die Top 10 als Texttabelle ausgeben
print("\n--- TOP 10 EINFLUSSREICHSTE FAKTOREN ---")
print(feature_imp_df.head(10).to_string(index=False))

## Kaskadierte Modellsystem
Auf Grundlage der Datenpunkte, die tatsächlich einen Personenschaden haben, wird ein zweites Modell trainiert, das nur noch zwischen Klasse 1 (Leicht), Klasse 2 (Schwer) und Klasse 3 (Tödlich) unterscheidet.

In [ ]:
# 1. Filtere NUR die Fälle mit Personenschaden (> 0) aus den Originaldaten
train_mask = (y_train.iloc[:, 0] > 0) if isinstance(y_train, pd.DataFrame) else (y_train > 0)
test_mask = (y_test.iloc[:, 0] > 0) if isinstance(y_test, pd.DataFrame) else (y_test > 0)

X_train_stage2 = X_train[train_mask]
X_test_stage2 = X_test[test_mask]

# Original-Zielvariablen holen
y_train_orig = y_train.iloc[:, 0] if isinstance(y_train, pd.DataFrame) else y_train
y_test_orig = y_test.iloc[:, 0] if isinstance(y_test, pd.DataFrame) else y_test

# Klassen ummappen von [1, 2, 3] auf [0, 1, 2] für XGBoost
y_train_stage2 = y_train_orig[train_mask] - 1
y_test_stage2 = y_test_orig[test_mask] - 1

print(f"Datenbasis für Modell 2 (nur Personenschäden):")
print(f"Trainingsdaten: {X_train_stage2.shape[0]} Fälle")
print(f"Testdaten: {X_test_stage2.shape[0]} Fälle\n")

# 2. Neue Klassengewichte für das 3-Klassen-Problem berechnen
from sklearn.utils.class_weight import compute_class_weight
classes_s2 = np.unique(y_train_stage2)
weights_s2 = compute_class_weight(class_weight='balanced', classes=classes_s2, y=y_train_stage2)
class_weights_dict_s2 = dict(zip(classes_s2, weights_s2))

print("Neue Gewichte für Stufe 2 (0=Leicht, 1=Schwer, 2=Tödlich):")
print(class_weights_dict_s2)

sample_weights_s2 = np.array([class_weights_dict_s2[cls] for cls in y_train_stage2])

# 3. Multiklassifikations-XGBoost für Stufe 2 definieren
stage2_model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    tree_method='hist',
    enable_categorical=True
)

# 4. Modell trainieren
print("\nStarte Training für Modell-Stufe 2...")
stage2_model.fit(X_train_stage2, y_train_stage2, sample_weight=sample_weights_s2)
print("Training Stufe 2 abgeschlossen.")

# 5. Vorhersagen & Evaluation
y_pred_stage2 = stage2_model.predict(X_test_stage2)

print("\n--- STAGE 2 CLASSIFICATION REPORT (NUR PERSONENSCHÄDEN) ---")
print(classification_report(y_test_stage2, y_pred_stage2, target_names=['Leicht (0)', 'Schwer (1)', 'Tödlich (2)']))

# 6. Normalisierte Konfusionsmatrix plotten
cm_s2 = confusion_matrix(y_test_stage2, y_pred_stage2, normalize='true')
plt.figure(figsize=(7, 5))
sns.heatmap(cm_s2, annot=True, fmt='.2f', cmap='Purples',
            xticklabels=['Pred Leicht', 'Pred Schwer', 'Pred Tödlich'],
            yticklabels=['True Leicht', 'True Schwer', 'True Tödlich'])
plt.ylabel('Tatsächliche Schwere')
plt.xlabel('Vorhergesagte Schwere')
plt.title('Normalized Confusion Matrix - Stufe 2 Modell')
plt.show()

## SHAP installieren und visualisieren
Die Modelle werden nun interpretiert und die Schwachstellen wissenschaftlich erkannt. Dafür wird SHAP auf beide Modelle angewendet.
Model 1:

In [ ]:
# 1. SHAP Bibliothek installieren (wichtig, falls noch nicht geschehen)
!pip install shap

import shap

# 2. SHAP Explainer für das binäre Modell (Modell 1) initialisieren
# TreeExplainer ist perfekt optimiert für XGBoost
explainer = shap.TreeExplainer(best_model)

# 3. SHAP-Werte für eine Stichprobe der Testdaten berechnen
# Wir nehmen 2000 Proben, damit die Berechnung in Colab nicht zu lange dauert
X_test_sample = X_test.sample(2000, random_state=42)
shap_values = explainer(X_test_sample)

# 4. Den SHAP Summary Plot zeichnen
plt.figure(figsize=(10, 8))
plt.title("SHAP Bee-Swarm Plot: Richtungseinfluss auf Personenschäden", fontsize=14)
shap.summary_plot(shap_values, X_test_sample)

Model 2:

In [ ]:
# 1. SHAP Explainer für das Stufe-2-Modell (Multiklassifikation) initialisieren
explainer_s2 = shap.TreeExplainer(stage2_model)

# 2. SHAP-Werte berechnen
# Wir nehmen wieder eine Stichprobe aus den Stufe-2-Testdaten
X_test_s2_sample = X_test_stage2.sample(min(2000, len(X_test_stage2)), random_state=42)
shap_values_s2 = explainer_s2(X_test_s2_sample)

# 3. Den SHAP Summary Plot für Klasse 0 (Leicht) zeichnen
# Bei Multiklassifikation sind die SHAP-Werte dreidimensional [Samples, Features, Klassen]
plt.figure(figsize=(10, 8))
plt.title("SHAP Bee-Swarm Plot: Risiko-Faktoren für LEICHTE Unfälle (Stufe 0)", fontsize=14)
shap.summary_plot(shap_values_s2[:, :, 0], X_test_s2_sample)

# 4. Den SHAP Summary Plot für Klasse 1 (Schwer) zeichnen
plt.figure(figsize=(10, 8))
plt.title("SHAP Bee-Swarm Plot: Risiko-Faktoren für SCHWERE Unfälle (Stufe 1)", fontsize=14)
shap.summary_plot(shap_values_s2[:, :, 1], X_test_s2_sample)

# 5. Den SHAP Summary Plot für Klasse 2 (Tödlich) zeichnen
# Index 2 entspricht der Klasse 2 (Tödlich)
plt.figure(figsize=(10, 8))
plt.title("SHAP Bee-Swarm Plot: Risiko-Faktoren für TÖDLICHE Unfälle (Stufe 2)", fontsize=14)
shap.summary_plot(shap_values_s2[:, :, 2], X_test_s2_sample)